[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/mlsysbook-vi-notebooks/blob/main/vol1-ch07-frameworks/03-vi-phan-tu-dong.ipynb)

# Vi phân tự động: băng, liên hợp và cái giá bộ nhớ

Huấn luyện cần gradient của một hàm mất mát vô hướng theo từng tham số. Vi phân tự động (automatic
differentiation) không làm phép tính vi phân nào mới: nó chỉ áp quy tắc dây chuyền (chain rule) cho
những phép sơ cấp đã biết đạo hàm, và việc của framework là ghi chép chuyện đó ở quy mô lớn. Sổ tay này
tự viết hai chế độ vi phân trong vài chục dòng, kiểm chúng với `torch.autograd`, rồi đo cái giá mà
chương nói là chi phối: bộ nhớ cho những giá trị phải giữ lại tới lượt ngược.

**Bạn sẽ làm:**
1. viết chế độ xuôi (forward mode) bằng số đối ngẫu, và chế độ ngược (reverse mode) bằng một cuộn băng,
   rồi chạy lại đúng vết tính tay $x^2 \sin x$ tại $x = 2$ của chương;
2. đếm chính xác số phép của hai chế độ khi số tham số tăng, rồi đo hai chế độ của PyTorch;
3. đọc đồ thị liên kết ngược mà PyTorch dựng, và đếm byte các giá trị nó lưu lại;
4. dựng lại phép tính bốn thành phần bộ nhớ huấn luyện của chương, và cạm bẫy tăng batch;
5. thử những cơ chế điều khiển luồng gradient, và cách viết hàm thuần khiết kiểu JAX.

In [ ]:
#@title Chuẩn bị: thư viện, hạt giống và hai hàm đối chiếu với sách { display-mode: "form" }
# Chạy ô này trước. Nó không cài thêm gì: mọi thư viện dùng ở đây đều có sẵn trên Colab.
import math, os, random, re, sys, time

import matplotlib.pyplot as plt
import numpy as np
import torch

# Số luồng CPU. 0 để thư viện tự chọn (trên Colab là 2).
NB_THREADS = int(os.environ.get("NB_THREADS", "0"))
if NB_THREADS:
    torch.set_num_threads(NB_THREADS)
    try:
        torch.set_num_interop_threads(NB_THREADS)
    except RuntimeError:  # chỉ đặt được một lần mỗi phiên
        pass
    from threadpoolctl import threadpool_limits
    threadpool_limits(NB_THREADS)

SEED = 42


def dat_hat_giong(seed=SEED):
    """Cùng hạt giống thì cùng kết quả, mỗi lần chạy lại."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


dat_hat_giong()
DATA = os.environ.get("NB_DATA", "data")  # nơi tải dữ liệu về
NB_STRICT = os.environ.get("NB_STRICT") == "1"
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.3})

# Đọc một con số theo cách bảng số liệu của bài học viết nó (dấu chấm hay dấu phẩy).
import math
import re

_SUP = str.maketrans("⁰¹²³⁴⁵⁶⁷⁸⁹⁻⁺", "0123456789-+")
_MULT = {"tỷ": 1e9, "tỉ": 1e9, "triệu": 1e6, "nghìn": 1e3, "ngàn": 1e3,
         "K": 1e3, "M": 1e6, "B": 1e9}
_NUM = re.compile(
    r"(?<![\w.,])(-?)(\d+(?:[.,]\d+)*)"                      # sign, digits with separators
    r"(?:\s*[×x·]\s*10\s*(?:\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))|[eE]([-+]?\d+)"
    r"|\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))?"                  # or a bare power, 10^{-5}
    r"(\s*%|\s*(?:tỷ|tỉ|triệu|nghìn|ngàn)(?!\w)|\s?[KMB](?![A-Za-z]))?")


def norm(text):
    """The outline's markup and typography reduced to plain text, as both sides compare it."""
    text = text.replace("{,}", ",").replace("**", "").replace("`", "")
    text = re.sub(r"[   ]", " ", text)
    text = re.sub(r"−(?=\d)", "-", text)
    text = re.sub(r"[~≈]", "", text)
    return re.sub(r"\s+", " ", text).strip()


def _readings(digits):
    """(value, decimals) for each way the separators in `digits` can be read."""
    out = []
    groups_ok = lambda parts: all(len(p) == 3 for p in parts[1:])
    for thou, dec in ((",", "."), (".", ",")):
        if thou in digits and dec in digits:
            head, _, tail = digits.rpartition(dec)
            if dec in head or not tail.isdigit() or not groups_ok(head.split(thou)):
                continue
            out.append((float(head.replace(thou, "") + "." + tail), len(tail)))
    for sep in (",", "."):
        other = "." if sep == "," else ","
        if sep in digits and other not in digits:
            parts = digits.split(sep)
            if groups_ok(parts):
                out.append((float("".join(parts)), 0))                    # thousands
            if len(parts) == 2:
                out.append((float(parts[0] + "." + parts[1]), len(parts[1])))  # decimal
    if "." not in digits and "," not in digits:
        out.append((float(digits), 0))
    return out


def candidates(text):
    """Every (value, half_unit) the numbers in `text` can mean.

    half_unit is half of the last printed digit, scaled like the value, so `437,5 KB`
    accepts 437.544. Each number is offered as written and, when a multiplier or `%`
    follows it, scaled by that too, so `8,5 tỉ` cites 8.5 or 8.5e9 and `20 %` cites 20 or 0.2.
    """
    out = []
    for m in _NUM.finditer(norm(text)):
        sign, digits, exp_caret, exp_sup, exp_e, pow_caret, pow_sup, suffix = m.groups()
        exp = exp_caret or (exp_sup.translate(_SUP) if exp_sup else None) or exp_e
        power = pow_caret or (pow_sup.translate(_SUP) if pow_sup else None)
        scale10 = 10.0 ** int(exp) if exp else 1.0
        suffix = (suffix or "").strip()
        extra = 0.01 if suffix == "%" else _MULT.get(suffix)
        for value, decimals in _readings(digits):
            if power:
                # 10^{-5}, 2³²: an exact value, so it is matched to 1 part in 10⁹; a looser
                # comparison needs an explicit tol.
                v = value ** int(power)
                half = abs(v) * 1e-9
            else:
                v = value * scale10
                half = 0.5 * 10.0 ** -decimals * scale10
            v = -v if sign else v
            out.append((v, half))
            if extra:
                out.append((v * extra, half * extra))
    return out


def match(value, text):
    """The (value, half_unit) reading of `text` that `value` equals, or None."""
    for v, half in candidates(text):
        if math.isclose(value, v, rel_tol=1e-9, abs_tol=1e-300):
            return v, half
    return None


def well_anchored(text):
    """False for a citation too bare to identify one cell, like "1" or "2"."""
    t = norm(text)
    rest = re.sub(r"[\d.,\s\-+]", "", t)
    if rest:
        return True
    sig = re.sub(r"[^\d]", "", t).lstrip("0")
    return len(sig) >= 2


def _so(x):
    """Một con số để in: số nguyên có dấu phẩy ngăn nghìn, số thực giữ 6 chữ số có nghĩa."""
    if isinstance(x, (int, np.integer)) or (isinstance(x, float) and x.is_integer() and abs(x) < 1e15):
        return f"{int(x):,}"
    return f"{x:,.6g}"


def sach(gia_tri, nguon=None, trich=None, tol=None):
    """Một con số của sách, kèm đúng chữ của nó: trong bảng số liệu của bài học (nguon)
    hoặc nguyên văn trong chương (trich)."""
    chu = nguon if nguon is not None else trich
    if match(gia_tri, chu) is None:
        raise ValueError(f"{gia_tri!r} không phải con số ghi trong {chu!r}")
    return gia_tri


def theo_sach(ten, gia_tri, sach, nguon=None, trich=None, tol=None):
    """So một con số tính được với con số của sách. In ✓ khi khớp, ✗ kèm độ lệch khi không."""
    chu = nguon if nguon is not None else trich
    khop = match(sach, chu)
    if khop is None:
        raise ValueError(f"{sach!r} không phải con số ghi trong {chu!r}")
    sai_so = tol if tol is not None else khop[1]
    dung = abs(gia_tri - sach) <= sai_so
    print(f"{'✓' if dung else '✗'} {ten}: tính được {_so(gia_tri)} · sách: {chu}")
    if not dung:
        loi = f"lệch {_so(gia_tri - sach)}, quá sai số cho phép {_so(sai_so)}"
        if NB_STRICT:
            raise AssertionError(f"{ten}: {loi}")
        print(f"   {loi}. Nếu bạn vừa đổi tham số ở trên thì đây là điều được chờ đợi.")
    # None, not the bool: a cell ending in theo_sach(...) would otherwise echo True under the ✓.


def do_tren_may(ten, gia_tri, don_vi=""):
    """Một con số ĐO trên máy đang chạy: mỗi máy một khác, không phải con số của sách."""
    print(f"⏱ {ten}: {_so(gia_tri)} {don_vi} (đo trên máy này)".replace("  ", " "))


print(f"Python {sys.version.split()[0]} · PyTorch {torch.__version__} · NumPy {np.__version__}"
      f" · {torch.get_num_threads()} luồng CPU")

In [ ]:
from collections import Counter

import pandas as pd
import torch.nn.functional as F
from torch import nn
from torch.func import functional_call, grad, jacfwd, vmap
from torch.utils.checkpoint import checkpoint_sequential

MAU = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]  # bốn màu cố định cho các chuỗi số liệu


def thoi_gian(viec, lan=5):
    """Thời gian nhỏ nhất của `lan` lần chạy (giây): ít bị nhiễu bởi việc khác trên máy nhất."""
    viec()
    tot = float("inf")
    for _ in range(lan):
        t0 = time.perf_counter()
        viec()
        tot = min(tot, time.perf_counter() - t0)
    return tot


print("sẵn sàng")

## 1. Chế độ xuôi: số đối ngẫu

Chế độ xuôi gắn mỗi con số với đạo hàm của nó theo một hướng đã chọn, gọi là số đối ngẫu. Mỗi phép tính
cả hai phần: phép nhân áp quy tắc nhân, $\sin$ áp quy tắc dây chuyền. Gieo mầm $\dot x = 1$ ở đầu vào
thì một lượt chạy cho ra cả $f(x)$ lẫn $f'(x)$.

Lớp `Doi` dưới đây cũng đếm số phép nó chạy, để phần 3 so chi phí hai chế độ.

In [ ]:
DEM = Counter()   # số lần gọi phép sơ cấp, theo từng chế độ


class Doi:
    """Số đối ngẫu: một giá trị và đạo hàm của nó theo một hướng đã gieo mầm."""

    __slots__ = ("v", "d")

    def __init__(self, v, d=0.0):
        self.v, self.d = v, d

    def __add__(self, o):
        DEM["xuôi"] += 1
        if isinstance(o, Doi):
            return Doi(self.v + o.v, self.d + o.d)
        return Doi(self.v + o, self.d)

    __radd__ = __add__

    def __mul__(self, o):
        DEM["xuôi"] += 1
        if isinstance(o, Doi):
            return Doi(self.v * o.v, self.d * o.v + self.v * o.d)   # quy tắc nhân
        return Doi(self.v * o, self.d * o)

    __rmul__ = __mul__

    def binh_phuong(self):
        DEM["xuôi"] += 1
        return Doi(self.v * self.v, 2 * self.v * self.d)

    def sin(self):
        DEM["xuôi"] += 1
        return Doi(math.sin(self.v), math.cos(self.v) * self.d)    # quy tắc dây chuyền

    def tanh(self):
        DEM["xuôi"] += 1
        t = math.tanh(self.v)
        return Doi(t, (1 - t * t) * self.d)


def f(x):
    """Hàm của chương: x bình phương nhân sin x, ba phép sơ cấp."""
    return x.binh_phuong() * x.sin()


X0 = sach(2.0, trich="with \\(x=2.0\\), we record")
ra = f(Doi(X0, 1.0))                       # gieo mầm: dx/dx = 1
dung_f = X0**2 * math.sin(X0)
dung_df = 2 * X0 * math.sin(X0) + X0**2 * math.cos(X0)
print(f"một lượt chạy xuôi: f(2) = {ra.v:.4f}, f'(2) = {ra.d:.4f}")
print(f"công thức đóng:     f(2) = {dung_f:.4f}, f'(2) = {dung_df:.4f}")
assert math.isclose(ra.v, dung_f) and math.isclose(ra.d, dung_df)

## 2. Chế độ ngược: một cuộn băng

Chế độ ngược có hai pha. Pha xuôi chạy phép tính như thường, và ghi mỗi kết quả lên một **băng**, kèm
các giá trị cha và đạo hàm cục bộ theo từng cha. Pha ngược gieo mầm liên hợp của đầu ra $\bar c = 1$,
rồi duyệt băng theo thứ tự ngược: mỗi đỉnh chuyển liên hợp của nó về các cha, nhân với đạo hàm cục bộ,
và **cộng dồn** vào cha. Phép cộng dồn ấy là chỗ một biến đi tới đầu ra bằng nhiều đường được tính đúng.

Thứ tự ghi lên băng là thứ tự topo của đồ thị, nên đi ngược băng là đi theo thứ tự topo ngược. Đó là
toàn bộ thuật toán.

In [ ]:
BANG = []   # băng: các đỉnh theo đúng thứ tự lượt xuôi tạo ra chúng


class Gia:
    """Một giá trị trên băng: nhớ các cha của nó và đạo hàm cục bộ theo từng cha."""

    __slots__ = ("v", "g", "cha", "dh", "ten")

    def __init__(self, v, cha=(), dh=(), ten=""):
        self.v, self.g, self.cha, self.dh, self.ten = v, 0.0, cha, dh, ten
        if cha:                           # tensor lá (tham số, đầu vào) không nằm trên băng
            BANG.append(self)
            DEM["ngược, lượt xuôi"] += 1

    def __add__(self, o):
        if isinstance(o, Gia):
            return Gia(self.v + o.v, (self, o), (1.0, 1.0))
        return Gia(self.v + o, (self,), (1.0,))

    __radd__ = __add__

    def __mul__(self, o):
        if isinstance(o, Gia):
            return Gia(self.v * o.v, (self, o), (o.v, self.v))
        return Gia(self.v * o, (self,), (o,))

    __rmul__ = __mul__

    def binh_phuong(self):
        return Gia(self.v * self.v, (self,), (2 * self.v,))

    def sin(self):
        return Gia(math.sin(self.v), (self,), (math.cos(self.v),))

    def tanh(self):
        t = math.tanh(self.v)
        return Gia(t, (self,), (1 - t * t,))

    def lan_truyen_nguoc(self, nhat_ky=None):
        """Gieo mầm liên hợp bằng 1, rồi duyệt băng theo thứ tự ngược."""
        self.g = 1.0
        for nut in reversed(BANG):
            for cha, dh in zip(nut.cha, nut.dh):
                cha.g += dh * nut.g           # cộng dồn: một biến có thể nhận từ nhiều đường
                DEM["ngược, lượt ngược"] += 1
                if nhat_ky is not None:
                    nhat_ky.append((nut.ten, cha.ten, dh, dh * nut.g, cha.g))


BANG.clear()
x = Gia(X0, ten="x")
a = x.binh_phuong()
a.ten = "a"
b = x.sin()
b.ten = "b"
c = a * b
c.ten = "c"
print(f"lượt xuôi ghi {len(BANG)} đỉnh lên băng: a = {a.v:.4f}, b = {b.v:.4f}, c = {c.v:.4f}")
nhat_ky = []
c.lan_truyen_nguoc(nhat_ky)
print("lượt ngược, theo thứ tự băng ngược:")
print(pd.DataFrame(nhat_ky, columns=["từ", "về", "đạo hàm cục bộ", "góp thêm", "liên hợp sau khi cộng"])
      .to_string(index=False, float_format="{:.4f}".format))

Đọc bảng từ trên xuống: phép nhân trao cho $a$ giá trị của $b$, và cho $b$ giá trị của $a$. Đường qua
$\sin$ góp $\bar b \cos x$ vào $\bar x$, đường qua phép bình phương góp $\bar a \cdot 2x$, và hai phần
cộng lại thành đạo hàm. Đó đúng là vết tính tay của chương, từng con số một. Ô dưới đối chiếu các giá trị
ghi trong chương, rồi so đạo hàm với `torch.autograd` và với lượt chạy xuôi ở phần 1.

In [ ]:
theo_sach("a = x²", a.v, sach=4.0, trich="\\(a=x^2=4.0\\)")
theo_sach("liên hợp của a", a.g, sach=0.9093, trich="\\(\\bar{a}=0.9093\\)")
theo_sach("liên hợp của b", b.g, sach=4.0, trich="\\(\\bar{b}=4.0\\)")
xt = torch.tensor(X0, dtype=torch.float64, requires_grad=True)
(xt**2 * torch.sin(xt)).backward()
print(f"đạo hàm: băng {x.g:.4f} · torch.autograd {xt.grad.item():.4f} · chế độ xuôi {ra.d:.4f} "
      f"· công thức đóng {dung_df:.4f}")
assert math.isclose(x.g, xt.grad.item()) and math.isclose(x.g, ra.d)

## 3. Cùng cuộn băng trên một MLP, và chi phí của hai chế độ

Cuộn băng không biết gì về mạng nơ-ron. Hàm dưới là một MLP 3 → H → 1 với tanh và mất mát bình phương,
viết bằng phép trên từng con số, và nó chạy được với cả `Doi` lẫn `Gia`. Trước hết ta kiểm gradient của
băng với `torch.autograd` trên cùng tham số.

In [ ]:
def mlp_vo_huong(w, x, y, H):
    """MLP 3 → H → 1, tanh, mất mát bình phương. w có 5H + 1 tham số: W1 (H×3), b1 (H), W2 (H), b2."""
    W1, b1, W2, b2 = w[:3 * H], w[3 * H:4 * H], w[4 * H:5 * H], w[5 * H]
    ra_ = b2
    for j in range(H):
        z = b1[j]
        for i in range(3):
            z = z + W1[3 * j + i] * x[i]
        ra_ = ra_ + W2[j] * z.tanh()
    sai = ra_ + (-y)
    return sai * sai


H = 4
dat_hat_giong()
w0 = torch.randn(5 * H + 1, dtype=torch.float64)
X_MAU, Y_MAU = [0.5, -1.2, 0.3], 0.7      # một mẫu ví dụ

BANG.clear()
w_bang = [Gia(float(v)) for v in w0]
mlp_vo_huong(w_bang, X_MAU, Y_MAU, H).lan_truyen_nguoc()
g_bang = torch.tensor([p.g for p in w_bang], dtype=torch.float64)

wt = w0.clone().requires_grad_(True)
h_t = torch.tanh(wt[:3 * H].view(H, 3) @ torch.tensor(X_MAU, dtype=torch.float64) + wt[3 * H:4 * H])
((wt[4 * H:5 * H] @ h_t + wt[5 * H] - Y_MAU) ** 2).backward()
print(f"{len(w_bang)} tham số, {len(BANG)} đỉnh trên băng")
print("sai khác lớn nhất giữa băng và torch.autograd:", f"{(g_bang - wt.grad).abs().max().item():.1e}")
assert torch.allclose(g_bang, wt.grad, rtol=1e-12, atol=1e-14)

Bây giờ đếm. Chế độ xuôi cho một đạo hàm theo **một** hướng mỗi lượt, nên muốn gradient đầy đủ theo $P$
tham số thì phải chạy $P$ lượt, mỗi lượt gieo mầm một tham số. Chế độ ngược chạy một lượt xuôi có ghi
băng và một lượt ngược, mỗi cung của đồ thị một phép nhân cộng. Ô dưới đếm số lần gọi phép sơ cấp của
cả hai khi chiều rộng H tăng. Một phép của số đối ngẫu làm nhiều việc hơn một phép thường, nên đây là số
lần gọi chứ không phải số phép tính dấu phẩy động.

**Dự đoán:** ở H = 64, tức 321 tham số, chế độ xuôi gọi nhiều hơn chế độ ngược bao nhiêu lần?

In [ ]:
hang = []
for H_ in (1, 4, 16, 64):
    P = 5 * H_ + 1
    w_np = torch.randn(P, dtype=torch.float64).tolist()
    DEM.clear()
    BANG.clear()
    w_bang = [Gia(v) for v in w_np]
    mlp_vo_huong(w_bang, X_MAU, Y_MAU, H_).lan_truyen_nguoc()
    n_phep, n_cung = DEM["ngược, lượt xuôi"], DEM["ngược, lượt ngược"]
    DEM.clear()
    g_xuoi = []
    for k in range(P):                   # mỗi tham số một hướng tiếp tuyến, mỗi hướng một lượt chạy
        w_doi = [Doi(v, 1.0 if i == k else 0.0) for i, v in enumerate(w_np)]
        g_xuoi.append(mlp_vo_huong(w_doi, X_MAU, Y_MAU, H_).d)
    assert DEM["xuôi"] == P * n_phep     # P lượt, mỗi lượt đúng các phép của một lượt xuôi
    assert np.allclose(g_xuoi, [p.g for p in w_bang])
    hang.append((H_, P, n_phep, DEM["xuôi"], n_phep + n_cung, DEM["xuôi"] / (n_phep + n_cung)))
print(pd.DataFrame(hang, columns=["H", "số tham số P", "phép mỗi lượt xuôi", "chế độ xuôi (P lượt)",
                                  "chế độ ngược (xuôi + ngược)", "xuôi / ngược"])
      .to_string(index=False, formatters={"chế độ xuôi (P lượt)": "{:,}".format,
                                          "xuôi / ngược": "{:.1f}×".format}))

Hai chế độ cho cùng một gradient, đã kiểm bằng `assert`. Chi phí của chế độ ngược tăng theo số phép của
đồ thị, còn chi phí của chế độ xuôi tăng theo số phép **nhân với** số tham số. Cột cuối vì vậy tăng gần
tỉ lệ với $P$.

## 4. Đo hai chế độ của PyTorch

`torch.func` có cả hai: `grad` là chế độ ngược, `jacfwd` là chế độ xuôi, gom $P$ hướng tiếp tuyến thành
một batch bằng `vmap`. Chương nói rõ cách gom đó không đổi bản chất: số hướng vẫn tỉ lệ với số chiều
đầu vào. Ta đo cả hai trên một MLP nhỏ 16 → H → 4 với batch 64, rồi tăng H.

In [ ]:
dat_hat_giong()
X_t, Y_t = torch.randn(64, 16), torch.randint(0, 4, (64,))
ket = []
for H_ in (2, 8, 32, 128):
    mo_hinh = nn.Sequential(nn.Linear(16, H_), nn.Tanh(), nn.Linear(H_, 4))
    ten_ts = [n for n, _ in mo_hinh.named_parameters()]
    hinh_ts = [p.shape for p in mo_hinh.parameters()]
    w_phang = torch.cat([p.detach().reshape(-1) for p in mo_hinh.parameters()])

    def L(w):
        """Mất mát như một hàm của một vector tham số phẳng."""
        ts, i = {}, 0
        for n, s in zip(ten_ts, hinh_ts):
            ts[n] = w[i:i + s.numel()].view(s)
            i += s.numel()
        return F.cross_entropy(functional_call(mo_hinh, ts, (X_t,)), Y_t)

    assert torch.allclose(grad(L)(w_phang), jacfwd(L)(w_phang), atol=1e-6)
    ket.append((w_phang.numel(), thoi_gian(lambda: grad(L)(w_phang), 3), thoi_gian(lambda: jacfwd(L)(w_phang), 3)))
for P, t_nguoc, t_xuoi in ket:
    do_tren_may(f"P = {P:>5,}  grad (chế độ ngược)", t_nguoc * 1e3, "ms")
    do_tren_may(f"P = {P:>5,}  jacfwd (chế độ xuôi)", t_xuoi * 1e3, "ms")

fig, ax = plt.subplots(figsize=(7, 3.8))
Ps = [k[0] for k in ket]
ax.loglog(Ps, [k[1] * 1e3 for k in ket], "o-", color=MAU[0], ms=8, label="grad: chế độ ngược")
ax.loglog(Ps, [k[2] * 1e3 for k in ket], "s--", color=MAU[1], ms=8, label="jacfwd: chế độ xuôi, P hướng gom thành batch")
ax.set_xlabel("số tham số P (thang log)")
ax.set_ylabel("thời gian một gradient (ms)")
ax.set_title("Hai chế độ vi phân trên cùng một mô hình (đo trên máy này)")
ax.legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

Phép đếm ở phần 3 nói đường của chế độ xuôi phải dốc lên theo $P$, còn đường của chế độ ngược chỉ tăng
theo kích thước đồ thị, nên thoải hơn nhiều. Biểu đồ cho bạn kiểm điều đó trên máy của mình. Đó là lý do
chương nói chế độ ngược là lựa chọn tiêu chuẩn khi có một mất mát vô hướng và rất nhiều tham số, còn chế
độ xuôi đáng dùng khi số hướng đầu vào nhỏ.

## 5. Đồ thị liên kết ngược của PyTorch

PyTorch không giữ một danh sách như `BANG`. Mỗi tensor sinh ra từ một phép khả vi giữ `grad_fn`, trỏ tới
đỉnh hàm đã tạo ra nó, và mỗi đỉnh trỏ về các đầu vào của nó qua `next_functions`. Chương minh hoạ bằng
$y = 3x$, $z = y^2$: đi từ $z$ về phải gặp `PowBackward0`, rồi `MulBackward0`, rồi `AccumulateGrad` của
tensor lá $x$, nơi gradient dừng lại và cộng dồn vào `.grad`.

In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = x * 3
z = y**2
chuoi_ham, fn = [], z.grad_fn
while fn is not None:
    chuoi_ham.append(type(fn).__name__)
    fn = fn.next_functions[0][0] if fn.next_functions else None
print(" → ".join(chuoi_ham))
assert chuoi_ham == ["PowBackward0", "MulBackward0", "AccumulateGrad"]

# MLP hai tầng, cùng hình dạng với bảng 16 ở sổ tay 01: batch 32, 784 → 256 → 10
dat_hat_giong()
xb, yb = torch.randn(32, 784), torch.randint(0, 10, (32,))
W1 = (torch.randn(784, 256) * 0.05).requires_grad_()
b1 = torch.zeros(256, requires_grad=True)
W2 = (torch.randn(256, 10) * 0.05).requires_grad_()
b2 = torch.zeros(10, requires_grad=True)
an = xb @ W1 + b1
kich_hoat = torch.relu(an)
mat_mat = F.cross_entropy(kich_hoat @ W2 + b2, yb)

hang, da_tham, hang_doi = [], set(), [mat_mat.grad_fn]
while hang_doi:
    fn = hang_doi.pop(0)
    if fn is None or fn in da_tham:
        continue
    da_tham.add(fn)
    giu = [f"{k[len('_saved_'):]} {tuple(v.shape)}" for k in dir(fn) if k.startswith("_saved_")
           and isinstance(v := getattr(fn, k), torch.Tensor) and v.dim() > 0]
    hang.append((type(fn).__name__, ", ".join(giu)))
    hang_doi += [n for n, _ in fn.next_functions]
print(pd.DataFrame(hang, columns=["đỉnh hàm", "tensor nó giữ cho lượt ngược"]).to_string(index=False))
assert sum(t == "AccumulateGrad" for t, _ in hang) == 4   # bốn tham số lá; xb không cần gradient

Mỗi đỉnh tự nó chỉ là con trỏ và siêu dữ liệu, nhưng cột bên phải là các tensor mà nó giữ: đầu vào của
phép nhân ma trận, đầu ra của ReLU. Chúng phải sống tới khi lượt ngược đi qua đỉnh đó. Chương gọi việc
giữ giá trị này là chi phí chi phối của cả hệ vi phân tự động.

## 6. Con sóng bộ nhớ, và lưu điểm kích hoạt

`torch.autograd.graph.saved_tensors_hooks` cho ta móc vào đúng hai khoảnh khắc: khi một phép lưu tensor
cho lượt ngược, và khi lượt ngược lấy nó ra. Hàm dưới đếm số byte còn sống sau mỗi khoảnh khắc. Hai quy
ước: tham số không tính, vì chúng sống sẵn trong mô hình; và một tensor được hai phép lưu (đầu ra ReLU
cũng là đầu vào của tầng sau) chỉ tính một lần, coi như giải phóng khi lần dùng cuối xong.

Lưu điểm kích hoạt (activation checkpointing) chỉ giữ giá trị ở ranh giới các đoạn, và tính lại phần
giữa trong lượt ngược. `checkpoint_sequential` chia mạng thành các đoạn như vậy.

**Dự đoán:** mạng 16 tầng chia 4 đoạn thì đỉnh sóng còn khoảng bao nhiêu phần so với khi lưu hết?

In [ ]:
def mang(so_tang, rong=256):
    lop = []
    for _ in range(so_tang):
        lop += [nn.Linear(rong, rong), nn.ReLU()]
    return nn.Sequential(*lop)


def song_bo_nho(mo_hinh, x, chay=None):
    """(đỉnh, dãy (lượt, byte còn sống)) của các giá trị được lưu, qua một lượt xuôi và một lượt ngược."""
    tham_so = {p.untyped_storage().data_ptr() for p in mo_hinh.parameters()}
    song, dang_song, tong = [], {}, [0]

    def luu(t):
        k = t.untyped_storage().data_ptr()        # theo vùng nhớ: một view không tốn thêm byte nào
        if k in tham_so or t.untyped_storage().nbytes() == 0:
            return (t, None)
        if k not in dang_song:
            dang_song[k] = [t.untyped_storage().nbytes(), 0]
            tong[0] += dang_song[k][0]
        dang_song[k][1] += 1
        song.append(("xuôi", tong[0]))
        return (t, k)

    def dung(goi):
        t, k = goi
        if k is not None:
            dang_song[k][1] -= 1
            if dang_song[k][1] == 0:
                tong[0] -= dang_song.pop(k)[0]
            song.append(("ngược", tong[0]))
        return t

    with torch.autograd.graph.saved_tensors_hooks(luu, dung):
        ra_ = (chay(x) if chay else mo_hinh(x)).sum()
    dinh = tong[0]
    ra_.backward()
    return dinh, song


dat_hat_giong()
mo_hinh = mang(16)
x64 = torch.randn(64, 256)
goi_xuoi = [0]
for m in mo_hinh:
    m.register_forward_hook(lambda *_: goi_xuoi.__setitem__(0, goi_xuoi[0] + 1))

dinh_het, song_het = song_bo_nho(mo_hinh, x64)
lan_dau = goi_xuoi[0]
goi_xuoi[0] = 0
dinh_ckpt, song_ckpt = song_bo_nho(mo_hinh, x64,
                                   lambda z: checkpoint_sequential(mo_hinh, 4, z, use_reentrant=False))
print(f"lưu hết:      đỉnh {dinh_het / 1e6:.2f} MB, {lan_dau} lần gọi module xuôi")
print(f"lưu điểm 4 đoạn: đỉnh {dinh_ckpt / 1e6:.2f} MB, {goi_xuoi[0]} lần gọi module xuôi "
      f"({goi_xuoi[0] - lan_dau} lần là tính lại trong lượt ngược)")
assert dinh_ckpt < dinh_het and goi_xuoi[0] > lan_dau

def truc_tien_trinh(song):
    """Đặt các lần lưu của lượt xuôi trên [0, 1] và các lần lấy ra của lượt ngược trên [1, 2]."""
    xuoi = [b for pha, b in song if pha == "xuôi"]
    nguoc = [b for pha, b in song if pha == "ngược"]
    x_ = np.concatenate([np.linspace(0, 1, len(xuoi) + 1)[1:], 1 + np.linspace(0, 1, len(nguoc) + 1)[1:]])
    return np.concatenate([[0], x_]), np.concatenate([[0], xuoi, nguoc]) / 1e6


fig, ax = plt.subplots(figsize=(7, 3.8))
ax.plot(*truc_tien_trinh(song_het), color=MAU[0], lw=2, label="lưu mọi giá trị")
ax.plot(*truc_tien_trinh(song_ckpt), color=MAU[1], lw=2, ls="--", label="lưu điểm, 4 đoạn")
ax.axvline(1, color="0.6", lw=1)
ax.annotate("đỉnh: lượt ngược bắt đầu", (1, dinh_het / 1e6), xytext=(8, -4), textcoords="offset points", fontsize=8)
ax.set_xticks([0, 1, 2], ["đầu lượt xuôi", "đầu lượt ngược", "cuối lượt ngược"])
ax.set_xlim(-0.1, 2.1)
nhan_truc = ax.get_xticklabels()
nhan_truc[0].set_ha("left")
nhan_truc[-1].set_ha("right")
ax.set_ylabel("giá trị đã lưu còn sống (MB)")
ax.set_title("Con sóng bộ nhớ của một mạng 16 tầng, batch 64")
ax.set_ylim(0, dinh_het / 1e6 * 1.15)
ax.legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

Sóng dâng dần qua lượt xuôi, mỗi tầng thêm một tensor, đạt đỉnh đúng lúc lượt ngược bắt đầu, rồi hạ dần
khi lượt ngược đi qua từng tầng. Với lưu điểm, đường đứt chỉ đếm những gì được giữ qua khoảng giữa hai
lượt. Các giá trị tính lại chỉ sống ngắn trong lượt ngược, mỗi lần một đoạn, nên không hiện trên đường
này; cái giá của chúng là số lần gọi module xuôi in ở trên. Theo ngôn ngữ định luật sắt (iron law) mà
chương dùng, lưu điểm tăng số hạng tính toán để giảm số hạng lưu lượng bộ nhớ.

## 7. Bốn thành phần bộ nhớ huấn luyện

Chương viết bộ nhớ huấn luyện thành bốn số hạng:
$M_{total} = M_{weights} + M_{gradients} + M_{optimizer} + M_{activations}$. Ta đo bốn số hạng ấy trên một
MLP 784 → 256 → 10 huấn luyện bằng Adam, cùng hình dạng như ở phần 5. Trọng số và gradient đếm từ tham
số, trạng thái bộ tối ưu (optimizer state) đếm từ bộ tối ưu sau một bước, giá trị kích hoạt đếm qua móc
lưu tensor.

In [ ]:
def bon_thanh_phan(B):
    dat_hat_giong()
    mh = nn.Sequential(nn.Linear(784, 256), nn.ReLU(), nn.Linear(256, 10))
    toi_uu = torch.optim.Adam(mh.parameters())
    xb_, yb_ = torch.randn(B, 784), torch.randint(0, 10, (B,))
    kich = song_bo_nho(mh, xb_, lambda z: F.cross_entropy(mh(z), yb_))[0]
    toi_uu.step()
    trong_so = sum(p.numel() * p.element_size() for p in mh.parameters())
    gradient = sum(p.grad.numel() * p.grad.element_size() for p in mh.parameters())
    trang_thai = sum(v.numel() * v.element_size() for s in toi_uu.state.values()
                     for k, v in s.items() if k != "step")
    return trong_so, gradient, trang_thai, kich


hang = []
for B in (32, 256, 2048):
    w_, g_, o_, a_ = bon_thanh_phan(B)
    hang.append((B, w_ / 1e6, g_ / 1e6, o_ / 1e6, a_ / 1e6, (w_ + g_ + o_ + a_) / w_))
print(pd.DataFrame(hang, columns=["batch", "trọng số (MB)", "gradient (MB)", "Adam (MB)", "kích hoạt (MB)",
                                  "tổng / trọng số"]).to_string(index=False, float_format="{:.3f}".format))
w_, g_, o_, a32 = bon_thanh_phan(32)
assert g_ == w_ and o_ == 2 * w_          # gradient bằng trọng số, Adam giữ hai mô-men FP32
moi_mau = a32 / 32
print(f"giá trị kích hoạt mỗi mẫu: {moi_mau / 1e3:.2f} KB, nên kích hoạt vượt ba thành phần kia "
      f"từ batch khoảng {(w_ + g_ + o_) / moi_mau:,.0f}")

Với mô hình nhỏ này, ba thành phần đầu cố định còn giá trị kích hoạt tăng theo batch. ResNet-50 thì đưa
những ảnh lớn qua rất nhiều tầng tích chập, và chương cho thấy ở đó kích hoạt áp đảo hẳn.

### Các con số của chương

In [ ]:
GB, MB = 1e9, 1e6
P_RESNET = sach(25.6e6, nguon="25,6M · 102,4 MB")
w_resnet = P_RESNET * 4 / MB                                      # FP32: 4 byte mỗi tham số
theo_sach("ResNet-50: trọng số FP32 (MB)", w_resnet, sach=102.4, nguon="25,6M · 102,4 MB")
theo_sach("ResNet-50: gradient (MB)", w_resnet, sach=102.4, nguon="102,4 MB · 204,8 MB")
theo_sach("ResNet-50: hai mô-men FP32 của Adam (MB)", 2 * w_resnet, sach=204.8, nguon="102,4 MB · 204,8 MB")
kich_thap = sach(8, trich="With 8 GB–12 GB of saved activations")
kich_cao = sach(12, trich="With 8 GB–12 GB of saved activations")
tong_thap = kich_thap + (w_resnet + w_resnet + 2 * w_resnet) / 1e3
tong_cao = kich_cao + (w_resnet + w_resnet + 2 * w_resnet) / 1e3
theo_sach("ResNet-50: tổng bốn thành phần, thấp (GB)", tong_thap, sach=8.4, nguon="8,4–12,4 GB · 82,1–121,2×")
theo_sach("ResNet-50: tổng bốn thành phần, cao (GB)", tong_cao, sach=12.4, nguon="8,4–12,4 GB · 82,1–121,2×")
theo_sach("so với riêng trọng số, thấp", tong_thap * 1e3 / w_resnet, sach=82.1, nguon="8,4–12,4 GB · 82,1–121,2×")
theo_sach("so với riêng trọng số, cao", tong_cao * 1e3 / w_resnet, sach=121.2, nguon="8,4–12,4 GB · 82,1–121,2×")

# Adam với trọng số FP16: hai trạng thái FP32 mỗi tham số
FP16, FP32 = 2, 4
theo_sach("Adam so với trọng số FP16", 2 * FP32 / FP16, sach=4, nguon="4× · 5×")
theo_sach("trọng số cộng trạng thái, so với trọng số FP16", (FP16 + 2 * FP32) / FP16, sach=5, nguon="4× · 5×")
P7 = sach(7e9, nguon="7 tỷ · 70 GB = 14 GB + 56 GB")
theo_sach("7 tỷ tham số: trọng số FP16 (GB)", P7 * FP16 / GB, sach=14, nguon="7 tỷ · 70 GB = 14 GB + 56 GB")
theo_sach("7 tỷ tham số: trạng thái Adam (GB)", P7 * 2 * FP32 / GB, sach=56, nguon="7 tỷ · 70 GB = 14 GB + 56 GB")
theo_sach("7 tỷ tham số: tổng (GB)", P7 * (FP16 + 2 * FP32) / GB, sach=70, nguon="7 tỷ · 70 GB = 14 GB + 56 GB")
P175 = sach(175e9, nguon="175B · 350 GB")
theo_sach("175 tỷ tham số: riêng trọng số FP16 (GB)", P175 * FP16 / GB, sach=350, nguon="175B · 350 GB")

### Thuế hành chính (Napkin Math 1.3)

Chương mở rộng phép tính tới một mô hình tỷ tham số: trọng số FP16 2 GB, và giá trị kích hoạt ước lượng
bằng $B \times N_L \times S \times d_{model} \times n_{saved} \times 2$ byte, với batch 32, 100 tầng,
chuỗi 1024 token, chiều ẩn 1024 và một tensor được lưu mỗi tầng.

In [ ]:
w_ty = sach(2, trich="Model weights: 2 GB.")
g_ty = sach(2, trich="Gradients: 2 GB (same size as weights)")
o_ty = sach(8, trich="Optimizer states (Adam): 8 GB")
B_ = sach(32, trich="For a batch size of 32 and a 100-layer network")
N_L = sach(100, trich="For a batch size of 32 and a 100-layer network")
S_ = sach(1024, trich="For a 1024-token sequence, 1024-wide hidden state, and 1 saved tensor per layer")
d_ = sach(1024, trich="For a 1024-token sequence, 1024-wide hidden state, and 1 saved tensor per layer")
n_luu = sach(1, trich="For a 1024-token sequence, 1024-wide hidden state, and 1 saved tensor per layer")
a_ty = B_ * N_L * S_ * d_ * n_luu * FP16 / GB
theo_sach("giá trị kích hoạt (GB)", a_ty, sach=6.7, trich="\\approx \\mathbf{6.7 GB}", tol=0.05)
theo_sach("Adam so với trọng số FP16", o_ty / w_ty, sach=4, nguon="4× · 5×")
theo_sach("thuế hành chính: gradient + trạng thái + kích hoạt (GB)", g_ty + o_ty + a_ty, sach=16.7,
          trich="16.7 GB beyond the weights", tol=0.05)
print(f"bộ nhớ huấn luyện so với riêng trọng số: {(w_ty + g_ty + o_ty + a_ty) / w_ty:.1f} lần")

## 8. Cạm bẫy: tăng batch mà không tính bộ nhớ

Batch lớn hơn chia phụ trội điều phối cố định cho nhiều mẫu hơn, nhưng cũng nhân phần bộ nhớ kích hoạt
phụ thuộc batch. Chương nêu hai con số: mô hình 7 tỷ tham số ở FP16 chiếm 14 GB trên một A100 80 GB, và
tăng batch từ 8 lên 32 làm phần kích hoạt phụ thuộc batch tăng gấp bốn. Ta đo phần thứ hai trên mạng 16
tầng ở phần 6.

In [ ]:
B_nho = sach(8, nguon="8 → 32")
B_lon = sach(32, nguon="8 → 32")
dinh_nho, _ = song_bo_nho(mo_hinh, torch.randn(B_nho, 256))
dinh_lon, _ = song_bo_nho(mo_hinh, torch.randn(B_lon, 256))
print(f"batch {B_nho}: {dinh_nho / 1e3:.0f} KB, batch {B_lon}: {dinh_lon / 1e3:.0f} KB, gấp {dinh_lon / dinh_nho:.0f} lần")
assert dinh_lon == (B_lon // B_nho) * dinh_nho      # mọi giá trị lưu ở đây đều có chiều batch

A100 = sach(80, nguon="14 GB · 71,9 GB · 80 GB")
w7 = sach(14, nguon="14 GB · 71,9 GB · 80 GB")
con_lai = sach(71.9, nguon="14 GB · 71,9 GB · 80 GB")
print(f"80 GB − 14 GB = {A100 - w7:.1f} GB, nếu cả hai cùng là GB thập phân")
print(f"80 GiB = {A100 * 2**30 / GB:.1f} GB, trừ 14 GB còn {A100 * 2**30 / GB - w7:.1f} GB: đây là cách ra {con_lai} GB")

Phép trừ của chương chỉ khép khi 80 là GiB (dung lượng nhớ ghi theo luỹ thừa của 2) còn 14 là GB thập
phân. Hai cách đọc lệch nhau gần 6 GB, nên khi tính ngân sách bộ nhớ, hãy đưa mọi số hạng về cùng một đơn
vị trước.

Với transformer, chương nói cơ chế tập trung (attention mechanism) còn thêm một số hạng tăng theo bình
phương độ dài chuỗi (sequence length). Ô dưới đếm byte được lưu của một lớp tập trung viết tay
$\mathrm{softmax}(QK^\top/\sqrt d)\,V$, vốn giữ cả ma trận $S \times S$, và của
`F.scaled_dot_product_attention`, khi S tăng gấp đôi.

In [ ]:
def byte_luu(ham, *dau_vao):
    da_luu = {}

    def luu(t):
        da_luu[t.untyped_storage().data_ptr()] = t.untyped_storage().nbytes()   # theo vùng nhớ, như trên
        return t

    with torch.autograd.graph.saved_tensors_hooks(luu, lambda t: t):
        ra_ = ham(*dau_vao)
    return sum(da_luu.values()), ra_


def tap_trung_tay(q, k, v):
    trong_so_tt = torch.softmax(q @ k.transpose(-1, -2) / math.sqrt(q.shape[-1]), dim=-1)
    return trong_so_tt @ v


B_tt, dau, d_dau = 4, 4, 32           # một lớp tập trung nhỏ, ví dụ
hang = []
for S in (64, 128, 256, 512):
    q, k, v = (torch.randn(B_tt, dau, S, d_dau, requires_grad=True) for _ in range(3))
    b_tay, o_tay = byte_luu(tap_trung_tay, q, k, v)
    b_sdpa, o_sdpa = byte_luu(F.scaled_dot_product_attention, q, k, v)
    assert torch.allclose(o_tay, o_sdpa, atol=1e-5)
    hang.append((S, b_tay / 1e6, b_sdpa / 1e6, B_tt * dau * S * S * 4 / 1e6))
print(pd.DataFrame(hang, columns=["S", "viết tay (MB)", "scaled_dot_product_attention (MB)", "một ma trận S×S FP32 (MB)"])
      .to_string(index=False, float_format="{:.2f}".format))

Cột cuối là kích thước của **một** ma trận trọng số tập trung, $B \times N_{heads} \times S^2$ giá trị:
nó gấp bốn mỗi khi S gấp đôi. Bản viết tay giữ đúng một ma trận như thế, đầu ra của softmax, cho lượt
ngược, cộng với Q, K, V vốn chỉ tăng tuyến tính, nên phần lưu của nó tiến dần tới tốc độ gấp bốn ấy. Nếu
bản `scaled_dot_product_attention` trên máy bạn chọn một kernel tiết kiệm bộ nhớ, nó không giữ ma trận
$S \times S$, và phần lưu chỉ tăng tuyến tính theo S. Đó là lý do chương nói các kernel tập trung tiết
kiệm bộ nhớ là quan trọng.

## 9. Điều khiển luồng gradient

Bảng 6 của chương liệt kê các cơ chế để can thiệp vào lượt ngược, mỗi cái kèm một rủi ro. Ô dưới chạy năm
cái, mỗi cái bằng một phép tính có đáp án biết trước.

In [ ]:
w = torch.tensor([1.0, 2.0], requires_grad=True)
x = torch.tensor([3.0, 4.0])

for _ in range(2):
    (w * x).sum().backward()                      # quên zero_grad() giữa hai lượt
print("1. tích lũy gradient: sau hai lượt ngược không đặt lại, w.grad =", w.grad.tolist())
assert torch.equal(w.grad, 2 * x)                 # hai bước bị trộn vào nhau, không có lỗi nào
w.grad = None

mm = (w * x).sum()
mm.backward()
try:
    mm.backward()                                 # các giá trị đã lưu đã được giải phóng
except RuntimeError as loi:
    print("2. lượt ngược thứ hai khi không giữ đồ thị:", str(loi).split(".")[0])
w.grad = None

(w.detach() * x).sum().add(w.sum()).backward()    # nhánh qua detach() không góp gì
print("3. detach(): gradient chỉ còn từ nhánh không bị cắt:", w.grad.tolist())
assert torch.equal(w.grad, torch.ones(2))
w.grad = None

moc = w.register_hook(lambda g: g.clamp(-1.0, 1.0))   # cắt gradient (gradient clipping) theo tensor
(w * x).sum().backward()
moc.remove()
print("4. móc cắt gradient về [-1, 1]:", w.grad.tolist())
assert torch.equal(w.grad, torch.ones(2))

a_ = torch.tensor([1.0, 2.0], requires_grad=True)
b_ = torch.tensor([3.0, 4.0], requires_grad=True)
y_ = (a_ * b_).sum()                              # phép nhân lưu b để tính gradient của a
b_.data.mul_(10)                                  # sửa qua .data: autograd không hay biết
y_.backward()
print("5. sửa qua .data sau lượt xuôi: a.grad =", a_.grad.tolist(), "thay vì [3.0, 4.0]")
assert torch.equal(a_.grad, torch.tensor([30.0, 40.0]))
a_ = torch.tensor([1.0, 2.0], requires_grad=True)
b_ = torch.tensor([3.0, 4.0], requires_grad=True)
y_ = (a_ * b_).sum()
with torch.no_grad():
    b_.mul_(10)                                   # sửa tại chỗ thông thường: bộ đếm phiên bản tăng
try:
    y_.backward()
except RuntimeError as loi:
    print("   sửa tại chỗ thông thường thì autograd bắt được:", str(loi).split(":")[0][:90])

Bốn trường hợp đầu là đúng như thiết kế. Gradient cộng dồn để cho phép tích lũy gradient (gradient
accumulation) qua nhiều batch, nên quên đặt lại thì trộn hai bước. Đồ thị được giải phóng ngay sau lượt
ngược trừ khi bạn yêu cầu giữ, và việc giữ tốn bộ nhớ. `detach()` cắt một cung của đồ thị, như khi đóng
băng các tầng đã huấn luyện trước trong học chuyển giao (transfer learning). Một móc chạy mã Python của
bạn trên tensor đã đăng ký, ở mọi lượt ngược. Trường hợp thứ năm là lý do chương cảnh báo về
`.data`: autograd đánh số phiên bản cho mỗi tensor để biết một giá trị đã lưu có bị sửa sau khi lưu hay
không, và `.data` đi vòng qua bộ đếm ấy. Kết quả sai mà không có lỗi nào.

Cơ chế cuối là **hàm autograd tự định nghĩa**: người viết tự nói phải lưu gì cho lượt ngược. Hàm mũ dưới
đây lưu **đầu ra** chứ không lưu đầu vào, vì đạo hàm của $e^x$ là chính nó. `gradcheck` so quy tắc tự viết
với sai phân hữu hạn.

In [ ]:
class Mu(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        y = torch.exp(x)
        ctx.save_for_backward(y)          # tự quyết định lưu gì: ở đây là đầu ra
        return y

    @staticmethod
    def backward(ctx, g):
        (y,) = ctx.saved_tensors
        return g * y


dat_hat_giong()
x_kt = torch.randn(6, dtype=torch.float64, requires_grad=True)
print("gradcheck:", torch.autograd.gradcheck(Mu.apply, (x_kt,)))
y_kt = Mu.apply(x_kt)
(da_luu,) = y_kt.grad_fn.saved_tensors
print("tensor được lưu chính là đầu ra:", da_luu.data_ptr() == y_kt.data_ptr())
assert da_luu.data_ptr() == y_kt.data_ptr()

## 10. Cùng một tầng, hai cách giữ trạng thái

Chương so ba framework trên cùng một mạng: một tầng tuyến tính nhận mười đầu vào và cho một đầu ra.
PyTorch gắn tham số vào đối tượng `nn.Module`. JAX coi mô hình là một hàm thuần khiết, tham số được đưa
vào tường minh, và vi phân là một **phép biến đổi hàm**: `grad(f)` trả về một hàm mới. `torch.func` cho
viết theo kiểu đó ngay trong PyTorch. Ghép `vmap` với `grad` cho gradient của **từng mẫu**, điều mà một
lượt `.backward()` thông thường không cho, vì nó cộng mọi mẫu vào một `.grad`.

In [ ]:
D_VAO, D_RA = 10, 1          # "a single linear layer mapping ten inputs to one output"
dat_hat_giong()
tang = nn.Linear(D_VAO, D_RA)
X_f, Y_f = torch.randn(16, D_VAO), torch.randn(16, D_RA)

# Kiểu có trạng thái: tham số nằm trong đối tượng, gradient đổ vào .grad
((tang(X_f) - Y_f) ** 2).mean().backward()
g_trang_thai = tang.weight.grad.clone()

# Kiểu hàm thuần khiết: tham số là dữ liệu, gradient là giá trị trả về
tham_so = {k: v.detach() for k, v in tang.named_parameters()}


def mat_mat_ham(ts, x, y):
    return ((functional_call(tang, ts, (x,)) - y) ** 2).mean()


ham_gradient = grad(mat_mat_ham)
print("grad(f) là:", type(ham_gradient).__name__, "→ gọi nó mới ra gradient")
g_ham = ham_gradient(tham_so, X_f, Y_f)
assert torch.allclose(g_ham["weight"], g_trang_thai)

g_tung_mau = vmap(ham_gradient, in_dims=(None, 0, 0))(tham_so, X_f, Y_f)
print("gradient của trọng số theo từng mẫu:", tuple(g_tung_mau["weight"].shape))
assert torch.allclose(g_tung_mau["weight"].mean(0), g_trang_thai, atol=1e-6)
print("trung bình gradient từng mẫu bằng gradient của cả batch: đúng")

Hai cách cho cùng một gradient, và `assert` đã kiểm. Khác nhau là chỗ trạng thái nằm: trong đối tượng, hay
trong tay người gọi. Chương nói cái giá của kiểu hàm thuần khiết là phải tự quản lý tham số; đổi lại, vì
hàm chỉ phụ thuộc đầu vào của nó, các phép biến đổi ghép được với nhau, như `vmap(grad(f))` vừa làm.

## Thử thêm

1. Ở ô `tape`, thay `c = a * b` bằng `c = a * b + a`, tức $x^2 \sin x + x^2$. Dự đoán liên hợp mới của
   $a$ trước khi chạy. `theo_sach` của $\bar a$ sẽ báo lệch vì đây không còn là hàm của chương, và
   `assert` cuối ô `tape_check` cũng dừng lại: hãy sửa dòng `torch.autograd` theo hàm mới.
2. Ở ô `wave`, đổi `checkpoint_sequential(mo_hinh, 4, ...)` thành 2 rồi 8 đoạn. Đỉnh sóng và số lần tính
   lại thay đổi theo hai chiều ngược nhau thế nào?
3. Ở ô `four_terms`, thay `torch.optim.Adam` bằng `torch.optim.SGD(mh.parameters(), lr=0.1)`. Cột Adam
   còn lại bao nhiêu, và `assert` nào dừng lại?

## Tóm lại

* Chế độ ngược là một cuộn băng: ghi lại khi chạy xuôi, duyệt ngược và cộng dồn liên hợp. Một lượt cho
  gradient theo mọi tham số, bất kể có bao nhiêu đường nối chúng tới mất mát.
* Chế độ xuôi cần một lượt cho mỗi hướng, nên muốn gradient đầy đủ thì chi phí tỉ lệ với số tham số.
* Cái giá của chế độ ngược là bộ nhớ: mọi giá trị được lưu sống tới khi lượt ngược dùng nó, tạo thành con
  sóng đạt đỉnh lúc lượt ngược bắt đầu. Lưu điểm kích hoạt đổi phép tính lại lấy đỉnh thấp hơn.
* Bộ nhớ huấn luyện là bốn số hạng: với ResNet-50 trong ví dụ của chương, tổng khoảng 82–121 lần riêng
  trọng số, và phần kích hoạt tăng theo batch.

---
### Nguồn và giấy phép

Sổ tay này đi kèm bài học [Framework — tầng giữa toán học và phần cứng](https://d3lu8vk4we0dc.cloudfront.net/lessons/vol1-ch07-frameworks/), dựng từ chương
[*ML Frameworks*](https://mlsysbook.ai/vol1/frameworks/frameworks.html) của sách. Một phần sổ tay phỏng theo phòng thí nghiệm `labs/vol1/lab_07_ml_frameworks.py` của sách (© 2026 President and Fellows of Harvard College, CC BY-NC-SA 4.0).

Nội dung gốc thuộc Machine Learning Systems của Vijay Janapa Reddi và cộng sự, MIT Press. Bản quyền © 2024-2026 Harvard University. Giấy phép CC BY-NC-SA 4.0. Bản tiếng Việt này là tác phẩm phái sinh dùng cùng giấy phép: ghi công, chia sẻ tương tự, phi thương mại.